### Source page 
https://cafef.vn/du-lieu/Lich-su-giao-dich-hrc-1.chn

In [1]:
import requests
import pandas as pd

# 1. Cấu hình Endpoint và Tham số
url = "https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx"

# Tham số (Query Parameters) tách riêng ra cho dễ quản lý
params = {
    "Symbol": "HRC",       # Mã cổ phiếu
    "StartDate": "",       # Để trống là lấy từ mới nhất
    "EndDate": "",
    "PageIndex": 4,        # Trang số 4 (như link bạn gửi)
    "PageSize": 20         # Lấy 20 dòng mỗi lần
}

# Giả lập trình duyệt (Header)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# 2. Gọi API
print("Đang tải dữ liệu JSON từ CafeF...")
response = requests.get(url, params=params, headers=headers)

# 3. Xử lý dữ liệu JSON
if response.status_code == 200:
    data_json = response.json()
    
    # Dữ liệu thực tế nằm sâu trong cấu trúc: Data -> Data (List)
    # Bạn nhìn file JSON sẽ thấy: {"Data": {"Data": [ ... ]}}
    raw_data = data_json['Data']['Data']
    
    # Đưa vào DataFrame
    df = pd.DataFrame(raw_data)
    
    # Hiển thị kết quả
    print(f"Đã lấy được {len(df)} dòng dữ liệu.")
    print(df[['Ngay', 'GiaDongCua', 'ThayDoi', 'KhoiLuongKhopLenh']].head())
else:
    print("Lỗi kết nối!")

Đang tải dữ liệu JSON từ CafeF...
Đã lấy được 20 dòng dữ liệu.
         Ngay  GiaDongCua         ThayDoi  KhoiLuongKhopLenh
0  26/11/2025       29.05   -0.9(-3.01 %)                300
1  25/11/2025       29.95  -0.35(-1.16 %)                300
2  24/11/2025       30.30       0(0.00 %)                  0
3  21/11/2025       30.30       1(3.41 %)                200
4  20/11/2025       29.30    0.35(1.21 %)                100


In [2]:
import requests
import pandas as pd
import time

def get_history_cafef_json(symbol):
    url = "https://cafef.vn/du-lieu/Ajax/PageNew/DataHistory/PriceHistory.ashx"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    all_data = []
    page = 1
    
    print(f"Bắt đầu crawl mã {symbol}...")
    
    while True:
        params = {
            "Symbol": symbol,
            "PageIndex": page,
            "PageSize": 1000, # MẸO: Xin server trả về 1000 dòng 1 lần cho nhanh
            "StartDate": "",
            "EndDate": ""
        }
        
        try:
            r = requests.get(url, params=params, headers=headers)
            data = r.json()
            
            # Lấy list bản ghi
            items = data['Data']['Data']
            
            # Nếu list rỗng nghĩa là đã hết dữ liệu -> Dừng lại
            if not items:
                break
                
            all_data.extend(items)
            print(f"Đã tải trang {page} - Tổng cộng: {len(all_data)} dòng")
            
            page += 1
            time.sleep(2) # Nghỉ chút để không bị chặn IP
            
        except Exception as e:
            print(f"Lỗi tại trang {page}: {e}")
            break
            
    # Tạo DataFrame tổng
    df = pd.DataFrame(all_data)
    
    # --- PHẦN LÀM SẠCH DỮ LIỆU (QUAN TRỌNG) ---
    # 1. Chuyển cột 'Ngay' sang datetime để sort
    df['Ngay'] = pd.to_datetime(df['Ngay'], format='%d/%m/%Y')
    
    # 2. Sắp xếp lại từ cũ đến mới
    df = df.sort_values('Ngay').reset_index(drop=True)
    
    # 3. Chọn các cột cần thiết
    cols = ['Ngay', 'GiaDieuChinh', 'GiaDongCua', 'KhoiLuongKhopLenh', 'GiaTriKhopLenh']
    return df[cols]

# Chạy thử
df_hrc = get_history_cafef_json('HRC')
print("\n--- KẾT QUẢ CUỐI CÙNG ---")
print(df_hrc.head())
print(df_hrc.tail())

Bắt đầu crawl mã HRC...
Đã tải trang 1 - Tổng cộng: 1000 dòng
Đã tải trang 2 - Tổng cộng: 2000 dòng
Đã tải trang 3 - Tổng cộng: 3000 dòng
Đã tải trang 4 - Tổng cộng: 4000 dòng
Đã tải trang 5 - Tổng cộng: 4776 dòng

--- KẾT QUẢ CUỐI CÙNG ---
        Ngay  GiaDieuChinh  GiaDongCua  KhoiLuongKhopLenh  GiaTriKhopLenh
0 2006-12-26         28.60       142.0              14150         2009300
1 2006-12-27         30.01       149.0                550           81950
2 2006-12-28         31.42       156.0               1100          171600
3 2006-12-29         32.82       163.0                100           16300
4 2007-01-02         34.44       171.0               6650      1137000000
           Ngay  GiaDieuChinh  GiaDongCua  KhoiLuongKhopLenh  GiaTriKhopLenh
4771 2026-02-23         37.05       37.05                800        29570000
4772 2026-02-24         39.60       39.60                700        27720000
4773 2026-02-25         42.35       42.35               2000        84700000
4774 20

In [3]:
## Save df_hrc 
df_hrc.to_csv("hrc_cafef.tsv", sep="\t")